# Experiment 034: declarative diagnostic (Colab)

This checks the failed smoke adapter on its exact declarative prompts and two
new declarative forms. It does not train a model or load MMLU.

## 0. Compute units and mount Drive

In [1]:
import glob, json, os, time
session_id = globals().get('session_id', time.strftime('%Y%m%d_%H%M%S'))
cu_before = 517.07
unit_rate = 6.77
print('session', session_id, '| cu_before =', cu_before if cu_before is not None else 'FILL ME')

from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/inm363'
RESULTS = os.path.join(ROOT, 'results')
OPEN_UNITS = os.path.join(RESULTS, 'open_cu_sessions')
UNITS_LOG = os.path.join(RESULTS, 'colab_units.jsonl')
os.makedirs(OPEN_UNITS, exist_ok=True)

old_sessions = [path for path in glob.glob(os.path.join(OPEN_UNITS, '*.json'))
                if not path.endswith(f'{session_id}.json')]
open_path = os.path.join(OPEN_UNITS, f'{session_id}.json')
if cu_before is None or unit_rate is None:
    print('fill cu_before and unit_rate, then rerun this cell')
else:
    for old_path in old_sessions:
        with open(old_path) as f:
            old = json.load(f)
        old['ts_end'] = time.strftime('%Y-%m-%d %H:%M')
        old['colab_cu_after'] = cu_before
        old['colab_cu_used'] = round(old['colab_cu_before'] - cu_before, 2)
        old['closed_on_resume'] = True
        with open(UNITS_LOG, 'a') as f:
            f.write(json.dumps(old) + '\n')
        os.remove(old_path)
        print('closed previous session:', old['session_id'])
    if not os.path.exists(open_path):
        record = {'session_id': session_id, 'ts_start': time.strftime('%Y-%m-%d %H:%M'),
                  'notebook': '034_1_colab_w2_diagnostic',
                  'colab_cu_before': cu_before, 'unit_rate': unit_rate}
        with open(open_path, 'w') as f:
            json.dump(record, f, indent=2)
        print('saved open session:', session_id)

session 20260815_214417 | cu_before = 517.07
Mounted at /content/drive
saved open session: 20260815_214417


## 1. GPU check

In [2]:
import subprocess, torch
gpu = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout
print(gpu.strip() or 'no gpu visible')
total_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'total vram: {total_gb:.1f} GiB')
assert 'A100' in gpu and total_gb >= 79, 'need the A100 80GB runtime'

GPU 0: NVIDIA A100-SXM4-80GB (UUID: GPU-ef0e96a1-10a1-4727-fbd1-ba85264f6a11)
total vram: 79.3 GiB


## 2. Clone or update the repo

In [3]:
import base64
import getpass
import glob
import os
from pathlib import Path
import subprocess

REPO_SLUG = 'hastov-lab/rl-grpo-or-sft-for-llms'
repo_url = f'https://github.com/{REPO_SLUG}.git'
repo_root = Path('/content') / REPO_SLUG.split('/')[-1]


def repo_git(*args):
    git = ['git']
    env = None
    try:
        from google.colab import userdata
    except ImportError:
        userdata = None

    # Private forks can use a read token from Colab Secrets.
    if userdata is not None and REPO_SLUG != 'hastov-lab/rl-grpo-or-sft-for-llms':
        try:
            token = userdata.get('GH_TOKEN')
        except Exception:
            token = None
        if isinstance(token, dict):
            token = token.get('value')
        if not isinstance(token, str) or not token.strip():
            token = getpass.getpass('GitHub token: ')
        if not token.strip():
            raise RuntimeError('This private fork requires a GitHub token with read access.')

        # Keep the token out of the command line and the saved remote URL.
        auth = base64.b64encode(f'x-access-token:{token.strip()}'.encode()).decode()
        env = dict(os.environ, REPO_GIT_AUTH=f'Authorization: Basic {auth}')
        git.append('--config-env=http.https://github.com/.extraHeader=REPO_GIT_AUTH')

    subprocess.run([*git, *args], env=env, check=True)


if repo_root.exists():
    if not (repo_root / '.git').exists():
        raise RuntimeError(f'Expected a Git checkout at {repo_root}.')
    origin = subprocess.check_output(
        ['git', '-C', str(repo_root), 'config', '--local', '--get', 'remote.origin.url'], text=True
    ).strip()
    if origin != repo_url:
        raise RuntimeError(f'The checkout at {repo_root} belongs to a different repository.')
    branch = subprocess.check_output(
        ['git', '-C', str(repo_root), 'branch', '--show-current'], text=True
    ).strip()
    if branch != 'main':
        raise RuntimeError(f'Expected the main branch at {repo_root}.')
    repo_git('-C', str(repo_root), 'pull', '--ff-only', 'origin', 'main')
else:
    repo_git('clone', '--branch', 'main', '--single-branch', '--depth', '1', repo_url, str(repo_root))

# Use the scripts from this checkout even if another experiment is open.
code_root = repo_root / '001_Code'
if not (code_root / 'scripts' / 'run.py').is_file():
    raise RuntimeError(f'Experiment scripts are missing from {code_root}.')
os.chdir(code_root)
print('working in:', os.getcwd())

working in: /content/INM363-IndividualProject-Dissertation/001_Code


## 3. Install dependencies

In [4]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'transformers==5.5.4', 'peft==0.19.1', 'accelerate==1.14.0',
                'datasets==4.8.4', 'pyyaml'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-q', '-y', 'torchao'], check=True)
print('dependencies installed')

dependencies installed


## 4. Session setup

In [5]:
import json, os, shutil, subprocess, sys, time
os.chdir(code_root)
sys.path.insert(0, os.getcwd())

from config import load_config, run_tag
from data.synthetic_facts import build_declarative_diagnostic
from file_utils import read_json
from obs import append_ledger, gpu_snapshot
from scripts.grpo_worker import stream_subprocess
from scripts.s12_eval import read_jsonl, sha256
from scripts.s28_compose import IntegrityError, adapter_hashes_of
from scripts.w2_diagnostic import (classify_diagnostic, commit_diagnostic_eval,
                                   diagnostic_state, load_diagnostic,
                                   write_diagnostic, write_diagnostic_summary)
from scripts.w2_probe import load_bundle, verify_start

CONFIG = 'configs/034_smoke_sft_colab_b15.yaml'
CFG = load_config(CONFIG)
assert cu_before is not None and unit_rate is not None, 'fill section 0 first'

SMOKE_ROOT = os.path.join(RESULTS, '034_smoke')
SOURCE_BUNDLE = os.path.join(SMOKE_ROOT, 'smoke_bundle.json')
START_RECORD = os.path.join(SMOKE_ROOT, 'start_seed0.json')
QUALIFICATION = os.path.join(SMOKE_ROOT, 'sft_qualification.json')
SFT_TAG = run_tag(CFG)
SFT_ADAPTER = os.path.join(ROOT, 'checkpoints', SFT_TAG, 'sft')
DIAG_ROOT = os.path.join(SMOKE_ROOT, 'declarative_diagnostic')
DIAGNOSTIC_PATH = os.path.join(DIAG_ROOT, 'prompt_bundle.json')
BASE_RESULT = os.path.join(DIAG_ROOT, 'baseline')
SFT_RESULT = os.path.join(DIAG_ROOT, 'sft')
SUMMARY_PATH = os.path.join(DIAG_ROOT, 'diagnostic_summary.json')
LEDGER = os.path.join(RESULTS, 'compute_ledger.jsonl')
os.makedirs(DIAG_ROOT, exist_ok=True)

assert os.path.exists(SOURCE_BUNDLE) and os.path.exists(START_RECORD)
assert read_json(QUALIFICATION)['passed'] is False, 'the original qualification must stay failed'
adapter_hashes_of(SFT_ADAPTER)
free_gb = shutil.disk_usage('/content/drive/MyDrive').free / 1024**3
print(f'drive free: {free_gb:.0f} GB')
assert free_gb >= 10, 'need at least 10GB free on Drive'

drive free: 178 GB


## 5. Wiring checks

In [6]:
import subprocess, sys
subprocess.run([
    sys.executable, '-m', 'pytest', '-q',
    'tests/test_synthetic_facts.py',
    'tests/test_w2_diagnostic.py',
    'tests/test_w2_probe.py',
    'tests/test_w2_worker.py',
], check=True)
print('wiring passed')

wiring passed


## 6. Freeze the diagnostic prompts

In [7]:
# freeze the prompt file before either checkpoint loads
source_bundle = load_bundle(SOURCE_BUNDLE)
diagnostic = build_declarative_diagnostic(source_bundle)
write_diagnostic(DIAGNOSTIC_PATH, diagnostic, source_bundle)
assert load_diagnostic(DIAGNOSTIC_PATH, source_bundle) == diagnostic
start = verify_start(CFG, START_RECORD)
assert diagnostic['source_dataset_sha256'] == start['dataset_sha256']
print('diagnostic:', diagnostic['sha256'])
print('rows:', len(diagnostic['rows']))

diagnostic: d44f8f0c5a3c60f028fbca3c1d064130d777174f64f44b3e825f4a376681d704
rows: 96


## 7. Evaluation helper

In [8]:
def free_vram_gib():
    value = subprocess.check_output([
        'nvidia-smi', '--query-gpu=memory.free', '--format=csv,noheader,nounits'
    ], text=True).strip().splitlines()[0]
    return float(value) / 1024

def run_diagnostic_branch(branch, result_dir, adapter_dir=None):
    state = diagnostic_state(result_dir, branch, DIAGNOSTIC_PATH,
                             START_RECORD, adapter_dir)
    if state == 'tampered':
        raise IntegrityError(f'{branch}: committed diagnostic is inconsistent')
    if state == 'committed':
        print(branch, 'diagnostic already committed')
        return

    scratch = f'/content/w2_declarative_{branch}_{session_id}'
    shutil.rmtree(scratch, ignore_errors=True)
    os.makedirs(result_dir, exist_ok=True)
    status_path = os.path.join(result_dir, 'diagnostic_status.json')
    if os.path.exists(status_path):
        old_status = read_json(status_path)
        if old_status.get('completed') is not True and old_status.get('error_type') not in (
                None, 'KeyboardInterrupt'):
            raise RuntimeError(f'{branch}: previous diagnostic failure needs review')
        archived_status = os.path.join(
            result_dir, f'diagnostic_status_{time.strftime("%Y%m%d_%H%M%S")}.json')
        os.replace(status_path, archived_status)
    cmd = [sys.executable, '-u', 'scripts/w2_diagnostic_worker.py',
           '--branch', branch, '--config', CONFIG,
           '--diagnostic', DIAGNOSTIC_PATH, '--source-bundle', SOURCE_BUNDLE,
           '--start-record', START_RECORD, '--scratch-dir', scratch,
           '--status', status_path]
    if adapter_dir:
        cmd.extend(['--adapter-dir', adapter_dir])

    started = time.time()
    completed = False
    status = {}
    try:
        free_gib = free_vram_gib()
        print(f'free vram before {branch}: {free_gib:.1f} GiB')
        assert free_gib >= 60, 'restart the runtime before the next diagnostic worker'
        proc = stream_subprocess(cmd)
        if os.path.exists(status_path):
            status = read_json(status_path)
        assert proc.returncode == 0, f'{branch}: worker returned {proc.returncode}'
        assert status.get('completed') is True, f'{branch}: worker did not complete'
        commit_diagnostic_eval(result_dir, scratch, branch, DIAGNOSTIC_PATH,
                               START_RECORD, adapter_dir, {'worker_status': status})
        completed = True
    finally:
        elapsed = round(time.time() - started, 1)
        gpu_row = status.get('gpu', gpu_snapshot())
        if status.get('peak_vram_gib') is not None:
            gpu_row['peak_vram_gb'] = status['peak_vram_gib']
        append_ledger(LEDGER, {
            'run_tag': '034_declarative_diagnostic_seed0',
            'condition': f'W2_declarative_{branch}',
            'timings_sec': {f'W2_declarative_{branch}': elapsed},
            'train_steps': 0, 'synthetic_passes': 2,
            'completed': completed, 'gpu': gpu_row,
        })
        shutil.rmtree(scratch, ignore_errors=True)

## 8. Baseline and failed endpoint

In [9]:
# commit each branch before the routing result is calculated
run_diagnostic_branch('baseline', BASE_RESULT)
run_diagnostic_branch('sft', SFT_RESULT, SFT_ADAPTER)
assert diagnostic_state(BASE_RESULT, 'baseline', DIAGNOSTIC_PATH,
                        START_RECORD) == 'committed'
assert diagnostic_state(SFT_RESULT, 'sft', DIAGNOSTIC_PATH,
                        START_RECORD, SFT_ADAPTER) == 'committed'

free vram before baseline: 79.2 GiB
Loading weights: 100% 338/338 [00:00<00:00, 2703.43it/s]
free vram before sft: 79.2 GiB
Loading weights: 100% 338/338 [00:00<00:00, 2565.60it/s]


## 9. Diagnostic result

In [10]:
base_generation = read_jsonl(os.path.join(BASE_RESULT, 'generation_preds.jsonl'))
base_continuation = read_jsonl(os.path.join(BASE_RESULT, 'continuation_preds.jsonl'))
sft_generation = read_jsonl(os.path.join(SFT_RESULT, 'generation_preds.jsonl'))
sft_continuation = read_jsonl(os.path.join(SFT_RESULT, 'continuation_preds.jsonl'))
summary = classify_diagnostic(
    base_generation, base_continuation, sft_generation, sft_continuation)
summary.update({
    'diagnostic_sha256': diagnostic['sha256'],
    'source_dataset_sha256': source_bundle['sha256'],
    'start_record_sha256': sha256(START_RECORD),
    'baseline_manifest_sha256': sha256(os.path.join(
        BASE_RESULT, 'diagnostic_eval_manifest.json')),
    'sft_manifest_sha256': sha256(os.path.join(
        SFT_RESULT, 'diagnostic_eval_manifest.json')),
    'sft_adapter_sha256': adapter_hashes_of(SFT_ADAPTER),
    'seed_bundle_reused': True,
    'repeated_inspection_disclosed': True,
})
write_diagnostic_summary(SUMMARY_PATH, summary)
print(json.dumps(summary, indent=2))
print('route:', summary['route'])
print('No repair is authorised by this result.')

{
  "completed": true,
  "post_hoc": true,
  "registered_w2_evidence": false,
  "independent_validation": false,
  "c1_claim_allowed": false,
  "repair_authorised": false,
  "thresholds": {
    "min_endpoint_accuracy": 0.75,
    "min_injected_gain": 0.25,
    "max_abs_control_change": 0.125
  },
  "values": {
    "generation": {
      "exact": {
        "baseline_injected": 0.0,
        "endpoint_injected": 1.0,
        "injected_gain": 1.0,
        "baseline_control": 0.0,
        "endpoint_control": 0.0,
        "control_change": 0.0,
        "passes_acquisition": true
      },
      "paraphrase": {
        "baseline_injected": 0.0,
        "endpoint_injected": 0.46875,
        "injected_gain": 0.46875,
        "baseline_control": 0.0,
        "endpoint_control": 0.0,
        "control_change": 0.0,
        "passes_acquisition": false
      }
    },
    "continuation": {
      "exact": {
        "baseline_injected": 0.3125,
        "endpoint_injected": 1.0,
        "injected_gain": 0.

## 10. Close the units record

In [12]:
import json, os, time
cu_after = 515.81
if cu_after is None:
    print('fill cu_after, then rerun this cell')
elif not os.path.isdir('/content/drive/MyDrive'):
    print('drive not mounted, run section 0 first')
else:
    open_path = os.path.join(OPEN_UNITS, f'{session_id}.json')
    if not os.path.exists(open_path):
        print('no open record for this session, it may already be closed')
    else:
        with open(open_path) as f:
            record = json.load(f)
        record['ts_end'] = time.strftime('%Y-%m-%d %H:%M')
        record['colab_cu_after'] = cu_after
        record['colab_cu_used'] = round(record['colab_cu_before'] - cu_after, 2)
        record['closed_on_resume'] = False
        with open(UNITS_LOG, 'a') as f:
            f.write(json.dumps(record) + '\n')
        os.remove(open_path)
        print('logged:', record)

logged: {'session_id': '20260815_214417', 'ts_start': '2026-08-15 21:44', 'notebook': '034_1_colab_w2_diagnostic', 'colab_cu_before': 517.07, 'unit_rate': 6.77, 'ts_end': '2026-08-15 22:01', 'colab_cu_after': 515.81, 'colab_cu_used': 1.26, 'closed_on_resume': False}
